# V-JEPA 2 — frame prediction demo

End-to-end on a single Colab T4 runtime:
1. Encode a small LIBERO Object slice with frozen V-JEPA 2 (ViT-L)
2. Train a tiny future-feature predictor (~30 sec)
3. For a held-out episode, predict features of frame `t+H` from frame `t`, retrieve the nearest real frame in the train set by cosine similarity
4. Plot the result: `input | true future | predicted (nearest neighbor)`

What this shows: the V-JEPA 2 representation is predictive — a simple MLP can forecast it H steps ahead, and the nearest real frame to the prediction is a plausible "what comes next".

Total compute on T4: ~5 min with defaults. **Runtime → Change runtime type → T4 GPU before running.**

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone repo and install deps

In [ ]:
%cd /content
!git clone -b feat/vjepa2-perception https://github.com/PoCInnovation/LeWM-Robot.git || (cd LeWM-Robot && git fetch && git checkout feat/vjepa2-perception && git pull)
%cd /content/LeWM-Robot/vjepa-perception

In [ ]:
!pip install -q -r requirements.txt

## 3. Settings

Bigger `MAX_EPISODES` = better demo but more GPU time. Defaults aim for ~5 min total on T4.

In [ ]:
MAX_EPISODES = 5      # source episodes to encode
FRAME_STRIDE = 2      # encode 1 frame out of N
BATCH_SIZE = 8        # encoder forward batch (lower if OOM)
HORIZON = 5           # how many cache steps ahead the predictor forecasts
VAL_EPISODES = 1      # trailing episodes held out for val + demo
EPOCHS = 30
NUM_DEMO_STEPS = 6    # rows in the demo grid

SRC_REPO = 'lerobot/libero_object_image'
CACHE_DIR = '/content/cached_features/libero_object'
CKPT_DIR = '/content/checkpoints/predictor'
DEMO_PNG = '/content/prediction_demo.png'

## 4. Precompute V-JEPA 2 features

Slowest cell: ~3-5 min on T4. Encodes the LIBERO frames into per-patch features and saves one `.safetensors` per episode.

In [ ]:
!python precompute_features.py \
    --src_repo "$SRC_REPO" \
    --dst_dir "$CACHE_DIR" \
    --vjepa2_repo facebook/vjepa2-vitl-fpc64-256 \
    --batch_size $BATCH_SIZE \
    --dtype float16 \
    --device cuda \
    --num_workers 2 \
    --max_episodes $MAX_EPISODES \
    --frame_stride $FRAME_STRIDE

In [ ]:
import json, pathlib
meta = json.loads((pathlib.Path(CACHE_DIR) / 'metadata.json').read_text())
print(f"episodes: {meta['num_episodes']}  frames cached: {meta['num_frames']}")
print(f"per-episode lengths: {meta['episode_lengths']}")

## 5. Train the future-feature predictor

MLP `(1024-dim → 512 → 512 → 1024)`. About 1M params. Trains in seconds because the frozen V-JEPA 2 features are already cached.

In [ ]:
!python train_predictor.py \
    --cache_dir "$CACHE_DIR" \
    --output_dir "$CKPT_DIR" \
    --horizon $HORIZON \
    --val_episodes $VAL_EPISODES \
    --epochs $EPOCHS \
    --batch_size 32 \
    --lr 1e-3

## 6. Generate the prediction demo grid

Re-instantiates `LeRobotDataset` (already cached locally from step 4) to grab RGB frames for visualization. For each chosen timestep `t` in the held-out episode, predicts features at `t+H`, finds the nearest real frame in the train set by cosine similarity, and stacks three columns:

`[ input frame_t | true frame_{t+H} | nearest-neighbor frame (model's prediction) ]`

In [ ]:
!python demo_prediction.py \
    --cache_dir "$CACHE_DIR" \
    --checkpoint "$CKPT_DIR/best.pt" \
    --src_repo "$SRC_REPO" \
    --num_steps $NUM_DEMO_STEPS \
    --output "$DEMO_PNG"

In [ ]:
from IPython.display import Image
Image(DEMO_PNG)

## 7. (Optional) Save artifacts to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/lewm_robot_demo
!cp -r "$CACHE_DIR" /content/drive/MyDrive/lewm_robot_demo/
!cp "$CKPT_DIR/best.pt" /content/drive/MyDrive/lewm_robot_demo/predictor_best.pt
!cp "$CKPT_DIR/history.json" /content/drive/MyDrive/lewm_robot_demo/predictor_history.json
!cp "$DEMO_PNG" /content/drive/MyDrive/lewm_robot_demo/
!ls -la /content/drive/MyDrive/lewm_robot_demo/